# Spatial Maps — DJF Percentiles and Event Occurrence

Produces gridded maps of:
1. Percentile threshold values (15th min-temp, 15th/85th precip).
2. Average annual occurrence of compound cold-dry (CD) and cold-wet (CW) events.

**Prerequisites:** `01_DJF_extraction.ipynb` must be run first.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patheffects as pe
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset

# ── User configuration ──────────────────────────────────────────────────────
percentiles_dir = "data/percentiles"
events_dir      = "data/percentiles"
haduk_sample    = ("/path/to/HadUK-Grid/5km/tasmin/day/latest/"
                   "tasmin_hadukgrid_uk_5km_day_19600101-19600131.nc")
fig_dir         = "figures"
N_YEARS         = 63
os.makedirs(fig_dir, exist_ok=True)

def getclosest_ij(lats, lons, latpt, lonpt):
    """Return (row, col) of the grid cell closest to (latpt, lonpt)."""
    dist_sq = (lats - latpt) ** 2 + (lons - lonpt) ** 2
    return np.unravel_index(dist_sq.argmin(), lats.shape)

CITIES = {
    "Glasgow":   (-4.263822, 55.864855),
    "Belfast":   (-5.929036, 54.594826),
    "Newcastle": (-1.601071, 54.978578),
    "Plymouth":  (-4.140809, 50.373450),
    "London":    (-0.121560, 51.511769),
}


## 1. Load coordinate grid and arrays

In [ ]:
ds = Dataset(haduk_sample, mode="r")
lat = ds.variables["latitude"][:]
lon = ds.variables["longitude"][:]
ds.close()

perc_minT     = np.load(os.path.join(percentiles_dir, "minT_percentiles_30yr.npy"))
perc_prec_15f = np.load(os.path.join(percentiles_dir, "perc15_percentiles_30yr_filtered.npy"))
perc_prec_85f = np.load(os.path.join(percentiles_dir, "perc85_percentiles_30yr_filtered.npy"))

months = ["dec", "jan", "feb"]
CD_totals, CW_totals = [], []
for m in months:
    CD_totals.append(np.load(os.path.join(events_dir, f"CD_event_counter_filtered_{m}_new.npy")))
    CW_totals.append(np.load(os.path.join(events_dir, f"CW_event_counter_filtered_{m}_new.npy")))

CD_all = np.array(CD_totals).sum(axis=0).astype(float)
CW_all = np.array(CW_totals).sum(axis=0).astype(float)
CD_all[CD_all < 1] = np.nan
CW_all[CW_all < 1] = np.nan
print("Data loaded.")


## 2. Map: average annual occurrence

In [ ]:
bins_cd = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
bins_cw = [0, 1, 2, 3]

fig, ax = plt.subplots(
    figsize=(10, 6), nrows=1, ncols=2, facecolor="white",
    subplot_kw={"projection": ccrs.PlateCarree()}
)

pcm  = ax[0].pcolormesh(lon, lat, CD_all / N_YEARS, cmap="tab20b_r")
pcm1 = ax[1].pcolormesh(lon, lat, CW_all / N_YEARS, cmap=plt.cm.get_cmap("tab20", 3))

for i, (panel, pcm_i, bins) in enumerate(
    zip(ax, [pcm, pcm1], [bins_cd, bins_cw])
):
    panel.set_extent([-11, 2, 49, 62])
    panel.add_feature(cfeature.COASTLINE, linewidth=0.5)
    panel.text(-10.5, 61, "AB"[i], transform=ccrs.PlateCarree(),
               fontsize=16, fontweight="bold")
    fig.colorbar(pcm_i, ax=panel, boundaries=bins, ticks=bins,
                 extend="max", orientation="horizontal",
                 fraction=0.046, pad=0.04)
    transform = ccrs.PlateCarree()._as_mpl_transform(panel)
    for city, (clon, clat) in CITIES.items():
        panel.plot(clon, clat, ".", color="k", transform=ccrs.PlateCarree())
        panel.annotate(city, xy=(clon, clat), xycoords=transform, color="white",
            path_effects=[pe.Stroke(linewidth=2, foreground="black"), pe.Normal()])

fig.text(0.5, 0.02, "Average number of compound cold events per winter",
         ha="center", fontsize=12)
fig.subplots_adjust(left=0.05, right=0.95, top=0.90, bottom=0.1, wspace=0.1)
fig.savefig(os.path.join(fig_dir, "CD_CW_occurrence_map.png"), dpi=300, facecolor="white")
plt.show()


## 3. Map: percentile threshold values

In [ ]:
perc_minT_p     = perc_minT.copy();     perc_minT_p[perc_minT_p > 500]       = np.nan
perc_prec_15f_p = perc_prec_15f.copy(); perc_prec_15f_p[perc_prec_15f_p > 5]  = np.nan
perc_prec_85f_p = perc_prec_85f.copy(); perc_prec_85f_p[perc_prec_85f_p > 500] = np.nan

tbins   = [-6, -4, -2, 0, 2, 4]
pbins15 = [0, 0.5, 1.0, 1.5, 2.0, 2.5, 3]
pbins85 = [0, 5, 10, 15, 20, 25, 30, 35, 40]

fig, ax = plt.subplots(
    figsize=(15, 5), nrows=1, ncols=3, facecolor="white",
    subplot_kw={"projection": ccrs.PlateCarree()}
)
specs = [
    (perc_minT_p,     "turbo",      9,  tbins,   tbins,   "Daily Minimum Temperature (°C)",
     r"15$^{th}$ Percentile of Minimum Temperature (°C)", "A"),
    (perc_prec_15f_p, "viridis_r",  None, pbins15, pbins15, "Daily Precipitation (mm)",
     r"15$^{th}$ Percentile of Precipitation (mm)", "B"),
    (perc_prec_85f_p, "viridis_r",  None, pbins85, pbins85, "Daily Precipitation (mm)",
     r"85$^{th}$ Percentile of Precipitation (mm)", "C"),
]
for i, (data, cmap_name, n, bnd, tks, cbar_lbl, title, letter) in enumerate(specs):
    cmap = plt.cm.get_cmap(cmap_name, n) if n else cmap_name
    pcm  = ax[i].pcolormesh(lon, lat, data, cmap=cmap)
    ax[i].set_extent([-11, 2, 49, 62])
    ax[i].set_title(title, fontsize=9)
    ax[i].add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax[i].text(-10.5, 61, letter, transform=ccrs.PlateCarree(), fontsize=16, fontweight="bold")
    fig.colorbar(pcm, ax=ax[i], extend="both" if i == 0 else "max",
                 boundaries=bnd, ticks=tks, label=cbar_lbl,
                 orientation="horizontal", fraction=0.046, pad=0.04)

fig.savefig(os.path.join(fig_dir, "percentile_thresholds_map.png"), dpi=300, facecolor="white")
plt.show()
